# Stable_Diffusion模型结构笔记

 -- Stable Diffusion 是 2022 年发布的文本到图像潜在扩散模型，基于 Latent Diffusion Models（LDMs）实现，由 CompVis、Stability AI 和 LAION 的研究人员创建。

-- 它能根据文本描述生成高质量图像，核心思想是利用文本信息指导纯噪声图片逐步去噪，以生成符合文本描述的图像。该模型可在消费级显卡上运行，应用广泛，涵盖图像生成、自然语言处理、音频视频生成等领域。

## 1. VAE（变分自编码器）使用L1损失
--功能：实现图像与低维隐空间的双向转换，核心是控制信息瓶颈（bottleneck=8通道）

--作用：负责图像的压缩和解码。在训练阶段，将高维图像数据压缩到低维潜空间，降低计算复杂度；在生成阶段，将潜空间的向量解码为最终的图像。

--结构与工作原理：由编码器和解码器组成。编码器将原始图像（如尺寸为 (n, c, h, w) ）编码至潜空间（一般为 (n, 4, 64, 64) ），解码器则将潜空间图像解码至普通图像。例如，将 512×512 像素的 RGB 图像（数据规模为 (3, 512, 512) ）压缩到潜空间（4, 64, 64），在潜空间进行扩散等操作后，再由解码器还原为图像。

In [ ]:
AutoencoderKL(
  encoder: 图像压缩（3通道→8通道隐空间）
    • 4级下采样：128→256→512→512通道
    • 残差块组(ResnetBlock2D) + 注意力模块(Attention)
    • 输出：8通道隐变量
  decoder: 隐空间重建（4通道→3通道图像）
    • 4级上采样：512→256→128通道
    • 对称残差结构
  quant_conv/post_quant_conv: 1x1卷积调整通道
)


## 2. CLIP Text文本编码系统
--作用：将输入的提示词转换为计算机可识别的文本向量，为图像生成提供语义信息。

--工作原理：采用预训练的 CLIP 模型（早期版本用 OpenAI 发布的预训练 ClipText 模型，V2 转向 OpenClip）。输入的文本提示语先分词，再输入 CLIP 的文本编码器，为每个 token 产生 768 维（1.x 版本）或 1024 维（2.x 版本）的向量。CLIP 模型是基于对比学习的多模态模型，通过在 4 亿个图片与标签文本对数据集上训练，学习图片与文本内容的对应关系。

--输出结果：输出一系列文本嵌入（ENCODER_HIDDEN_STATES），用于引导图像生成。例如，输入 “一只猫在草地上玩耍”，会得到包含该文本语义信息的向量

In [ ]:
CLIPTextTransformer(
  embeddings: 
    • token_embedding: 49408词表→1024维向量
    • position_embedding: 77位置编码
  encoder: 23层Transformer
    • 多头注意力(CLIPAttention) + LayerNorm
    • MLP扩展比4:1（1024→4096→1024）
)

# Tokenizer:
# - 词表大小：49408
# - 特殊标记：<|startoftext|>, <|endoftext|>, !(pad)
# - 固定序列长度：77

## 3. U-Net（噪声预测核心）
--作用：是扩散模型的核心，负责在潜空间对图像进行降噪推理，根据文本向量逐步去除噪声，生成高质量图像。

--工作原理：在预测过程中，反复调用 U - Net 迭代降噪。以文本向量为条件，从纯噪声数据开始，每次迭代预测并去除噪声 slice，逐渐生成符合文本描述的图像。如给定 “一个美丽的花园” 的文本向量和初始噪声，U - Net 不断调整噪声，使图像逐渐呈现出花园的特征。

--结构：采用 Encoder - Decoder 结构，包含 InputBlock 和 OutputBlock，主要由 ResBlock 和 AttentionBlock 构成。InputBlock 通过循环迭代架构，插入 ResBlock 和 AttentionBlock（或 SpatialTransformer），每个 input_block 有时间步嵌入参数，用于区分隐变量层次和指导扩散。OutputBlock 通过连接 input_block 的中间输出来防止梯度消失。

--关键技术 - 交叉注意力机制：贯穿整个 UNet 结构，使 UNet 中的每个空间位置都能 “注意” 到文字条件中不同的 token，获取文本提示语中不同位置的相互关联信息，实现以文本为条件的图像生成定向引导。

--核心机制：
- 文本条件注入：通过attn2模块将CLIP文本特征与图像特征对齐
- 残差连接：每级输出与上采样同尺度特征拼接
- 时间步控制：所有残差块包含time_emb_proj线性层


In [ ]:
UNet2DConditionModel(
  conv_in: 4通道输入 → 320通道
  time_embedding: 时间步编码（线性层+SiLU）
  down_blocks(4级下采样):
    • CrossAttnDownBlock2D(3个): 文本-图像跨注意力
      - Transformer2DModel: 文本key/value注入图像特征
      - 残差块 + 下采样卷积(stride=2)
    • DownBlock2D(1个): 纯卷积下采样
  up_blocks(4级上采样):
    • 对称跨注意力结构(CrossAttnUpBlock2D)
    • 特征拼接(skip connection)实现细节重建
  mid_block: 中心Transformer加强文本融合
)



## 4. 调度器（PNDMScheduler）
Scheduler（调度器）：配合 U - Net 工作，控制扩散过程中噪声添加和去除的节奏，决定每次迭代的降噪程度，对生成图像的质量和生成速度有重要影响。不同的调度器算法会导致不同的生成效果和效率。

In [ ]:
{
  "beta_schedule": "scaled_linear",  # 噪声调度策略
  "num_train_timesteps": 1000,       # 扩散步数
  "prediction_type": "epsilon"       # 预测噪声（非原始像素）
}


特点：Pruned DDIM加速采样，跳过部分扩散步（skip_prk_steps=True）

## 5. 辅助模块
- Feature Extractor (CLIPImageProcessor):
    - 预处理：224x224中心裁剪 + RGB转换
    - 归一化：ImageNet均值/std
- Safety Checker：未启用（通常用于内容过滤）

## 关键创新点总结
- 跨模态对齐：CLIP文本编码器与U-Net的CrossAttn模块实现文本-图像语义融合
- 分层条件控制：时间步嵌入(time_embedding)逐层调节扩散过程
- 高效采样：PNDM调度器在1000步训练基础上实现50步内高质量生成
- 隐空间优化：VAE的8通道瓶颈平衡重建质量与计算效率